# 01 — Generate the Logistics Q&A Dataset

Generates ~12,000 question/answer pairs using the Anthropic API.

**Before you run this notebook:**
1. Get an Anthropic API key from https://console.anthropic.com/
2. **Set a monthly spending cap** on the Anthropic console (Settings → Billing → Usage limits). Even $50 is enough; this run costs ~$5-15.
3. In Colab: Tools → User data → Add `ANTHROPIC_API_KEY`.

**Runtime:** ~30 minutes, costs ~$5-15 in Anthropic credits depending on model.

## Setup

In [2]:
# Clone the repo and install deps
!git clone https://github.com/masonsau0/logistics-qa-lora.git
%cd logistics-qa-lora
!pip install -q anthropic datasets python-dotenv

Cloning into 'logistics-qa-lora'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 57 (delta 9), reused 24 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 50.81 KiB | 3.18 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/logistics-qa-lora
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 25.5 MB/s eta 0:00:00


In [3]:
# Load API key from Colab secrets (Tools → User data)
import os

from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
assert os.environ["ANTHROPIC_API_KEY"].startswith("sk-ant-"), "Key looks malformed"

## Smoke test — verify the pipeline before spending money

In [4]:
# Generates ~50 examples (1 batch per category) — should finish in ~1 minute and cost < $0.10
!python -m data.prepare_dataset --smoke --batch-size 8

2026-06-02 23:01:41,530 | INFO | [freight_calculations] have=0, need=8
2026-06-02 23:02:01,416 | INFO | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-02 23:02:01,439 | INFO | [freight_calculations] +8 (total new this run: 8)
2026-06-02 23:02:01,439 | INFO | [carrier_performance] have=0, need=8
2026-06-02 23:02:30,717 | INFO | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-02 23:02:30,720 | INFO | [carrier_performance] +8 (total new this run: 16)
2026-06-02 23:02:30,720 | INFO | [claims_and_damages] have=0, need=8
2026-06-02 23:02:53,629 | INFO | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-02 23:02:53,630 | INFO | [claims_and_damages] +8 (total new this run: 24)
2026-06-02 23:02:53,631 | INFO | [routing_and_dispatch] have=0, need=8
2026-06-02 23:03:24,663 | INFO | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-02 23:03:24,664 | ERROR | [rout

In [5]:
# Inspect a few generated examples
import json

with open("data/raw_generated.jsonl") as f:
    for line in list(f)[:3]:
        rec = json.loads(line)
        print(f"[{rec['category']}]")
        print(f"Q: {rec['question']}")
        print(f"A: {rec['answer'][:200]}...")
        print(f"Key facts: {rec['key_facts']}")
        print()

[freight_calculations]
Q: A shipment measures 48 inches long, 40 inches wide, and 36 inches tall and weighs 800 pounds. Calculate the dimensional weight and determine whether actual weight or DIM weight should be used for rating.
A: To calculate dimensional weight, divide the volume by the DIM divisor. Volume = 48 × 40 × 36 = 69,120 cubic inches. Using the standard DIM divisor of 166 for LTL freight: 69,120 ÷ 166 = 416.39 pounds....
Key facts: ['DIM divisor 166', 'volume calculation', 'actual weight 800 lbs', 'compare DIM to actual', 'use greater weight']

[freight_calculations]
Q: What freight class would you assign to office furniture with a density of 12 pounds per cubic foot, and why does this matter for LTL pricing?
A: Office furniture with a density of 12 pounds per cubic foot typically falls into NMFC Class 70. Freight class is determined primarily by density, along with stowability, handling, and liability. Class...
Key facts: ['NMFC Class 70', 'office furniture', 'density 12 l

## Full run

Generates the full 12K-example dataset and writes the train/val/test splits.

Resumable — if Colab disconnects, just rerun. Already-generated examples are skipped.

In [6]:
!python -m data.prepare_dataset --target 6000 --batch-size 8 --split

2026-06-02 23:11:06,326 | INFO | Resuming: 48 existing records, skipping duplicates
2026-06-02 23:11:07,113 | INFO | [freight_calculations] have=8, need=1992
2026-06-02 23:11:28,648 | INFO | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-06-02 23:11:28,672 | INFO | [freight_calculations] +8 (total new this run: 8)
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/logistics-qa-lora/data/prepare_dataset.py", line 363, in <module>
    sys.exit(main())
             ^^^^^^
  File "/content/logistics-qa-lora/data/prepare_dataset.py", line 318, in main
    response_text = call_model_with_retry(client, args.model, prompt)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/logistics-qa-lora/data/prepare_dataset.py", line 137, in call_model_with_retry
    response = client.messages.create(
               ^^^^^^^^^^^^^^^^^^

In [ ]:
# Verify split sizes
for split in ["train", "val", "test"]:
    n = sum(1 for _ in open(f"data/{split}.jsonl"))
    print(f"{split}: {n}")

## Save to Google Drive

Mount Drive and copy the splits so they survive between Colab sessions.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/logistics-qa-lora/data
!cp data/*.jsonl /content/drive/MyDrive/logistics-qa-lora/data/
!ls -lh /content/drive/MyDrive/logistics-qa-lora/data/

### Next step
Open `02_train_lora.ipynb`.